# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 16 — Bảng kết quả khảo sát và proof

    Chỉ Run All khi tất cả suite đã chọn hoàn tất. Khóa model/config/ngưỡng trước test; không tuning trên test ở đây. Base month7 đã được quan sát ở các notebook cũ nên là development test, không gọi là tập xác nhận chưa từng mở. Bảng tách từng backbone, giữ kết quả thực tế kể cả phương pháp không tốt hơn baseline.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Mode: synthetic
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317]


In [3]:
for suite in PLAN["suites"]:
    marker = CAMPAIGN / "suites" / suite / "FROZEN.json"
    if not marker.is_file():
        raise RuntimeError(f"Chưa xong suite {suite}; hoàn tất notebook tương ứng trước16.")
    print("FROZEN:", suite)


FROZEN: losses
FROZEN: meta
FROZEN: lsr
FROZEN: xgboost


## 1. Đánh giá và bootstrap trong notebook thực thi riêng

    Lưu score test từng model, ngưỡng validation đã khóa, FPR thực tế/fairness và paired-row bootstrap ΔAP. Cùng hàng bootstrap cho mọi cặp seed. CI có điều kiện trên model đã fit; SD seed không thay thế CI test. Rerun chỉ kiểm và đọc kết quả đã hoàn tất.

In [4]:
EVALUATION = gru.research_launch_evaluation(CAMPAIGN)
print("Evaluation:", EVALUATION)


BEGIN_STAGE 1
ENTIRE_RESEARCH_TABLE_SEALED_BEFORE_TEST 40
RESEARCH_STUDY losses mode= synthetic device= cpu
TEST_METRICS {'suite': 'losses', 'method': 'balanced', 'seed': 42, 'comparison_group': 'shared_mlp', 'threshold': inf, 'fit_seconds': 0.21778714300035062, 'mode': 'synthetic', 'ap': 0.13569502613101272, 'roc_auc': 0.43355119825708066, 'recall_at_fpr5_curve': 0.0, 'recall_frozen_threshold': 0.0, 'fpr_frozen_threshold': 0.0, 'precision_frozen_threshold': 0.0, 'alert_rate_frozen_threshold': 0.0, 'recall_at_0p5': 0.0, 'precision_at_0p5': 0.0, 'f1_weighted_at_0p5': 0.7467289719626168, 'accuracy_at_0p5': 0.7833333333333333, 'tn': 102, 'fp': 0, 'fn': 18, 'tp': 0, 'younger_fpr': 0.0, 'younger_tpr': 0.0, 'younger_n': 63, 'older_fpr': 0.0, 'older_tpr': 0.0, 'older_n': 57, 'fpr_gap_older_minus_younger': 0.0, 'fpr_ratio_min_max': 1.0}
TEST_METRICS {'suite': 'losses', 'method': 'balanced', 'seed': 317, 'comparison_group': 'shared_mlp', 'threshold': 0.5451877117156982, 'fit_seconds': 0.2993535

## 2. Bảng theo phương pháp và paper

    Không ghép AP của notebook07/08/09 vào đây vì split/preprocessing/backbone/meta holdout khác. Cột adaptation phải đi cùng kết quả. Đọc PDF để bổ sung tập dữ liệu/kết quả gốc và khoảng trống; bảng số liệu tự nó chưa chứng minh tính mới.

In [5]:
import pandas as pd

table = pd.read_csv(EVALUATION / "research_table.csv")
paired = pd.read_csv(EVALUATION / "paired_bootstrap.csv")
display(table)
display(paired)

paper_table = gru.research_paper_table(CAMPAIGN)
display(paper_table)
print("CSV:", EVALUATION / "paper_research_table.csv")


CSV: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/evaluation/paper_research_table.csv


,comparison_group,method,ap_mean,ap_std,recall_frozen_threshold_mean,recall_frozen_threshold_std,fpr_frozen_threshold_mean,fpr_frozen_threshold_std,precision_frozen_threshold_mean,precision_frozen_threshold_std,older_fpr_mean,older_fpr_std,younger_fpr_mean,younger_fpr_std,fpr_gap_older_minus_younger_mean,fpr_gap_older_minus_younger_std,fpr_ratio_min_max_mean,fpr_ratio_min_max_std,fit_seconds_mean,fit_seconds_std
0,lsr_latent_mlp,lsr_nn_balanced,0.302812,0.047124,0.138889,0.117851,0.083333,0.103986,0.350000,0.212132,0.067308,0.095187,0.10,0.113137,-0.032692,0.017950,0.373932,0.528819,0.792004,0.020533
1,lsr_latent_mlp,lsr_nn_none,0.193465,0.079187,0.000000,0.000000,0.004902,0.006932,0.000000,0.000000,0.009615,0.013598,0.00,0.000000,0.009615,0.013598,0.500000,0.707107,0.734251,0.007685
2,lsr_latent_mlp,lsr_nn_weighted,0.223602,0.039173,0.000000,0.000000,0.009804,0.013865,0.000000,0.000000,0.019231,0.027196,0.00,0.000000,0.019231,0.027196,0.500000,0.707107,0.792903,0.131122
3,lsr_raw_lightgbm,lsr_lgbm_balanced,0.275245,0.000872,0.138889,0.196419,0.049020,0.027730,0.208333,0.294628,0.057692,0.027196,0.04,0.028284,0.017692,0.001088,0.650000,0.183848,0.043765,0.004256
4,lsr_raw_lightgbm,lsr_lgbm_none,0.279089,0.012928,0.083333,0.117851,0.029412,0.041595,0.166667,0.235702,0.019231,0.027196,0.04,0.056569,-0.020769,0.029372,0.740385,0.367152,0.076057,0.065393
5,lsr_raw_lightgbm,lsr_lgbm_weighted,0.279089,0.012928,0.083333,0.117851,0.029412,0.041595,0.166667,0.235702,0.019231,0.027196,0.04,0.056569,-0.020769,0.029372,0.740385,0.367152,0.035934,0.004332
6,normalized_head_mlp,ldam,0.249044,0.027038,0.055556,0.078567,0.029412,0.041595,0.125000,0.176777,0.048077,0.067991,0.01,0.014142,0.038077,0.053849,0.604000,0.560029,0.250167,0.015458
7,normalized_head_mlp,ldam_drw,0.231495,0.025391,0.138889,0.039284,0.088235,0.041595,0.225000,0.035355,0.125000,0.067991,0.05,0.014142,0.075000,0.053849,0.433333,0.122565,0.251896,0.004322
8,normalized_head_mlp,norm_ce,0.327481,0.056796,0.194444,0.039284,0.034314,0.006932,0.500000,0.000000,0.009615,0.013598,0.06,0.028284,-0.050385,0.041882,0.240385,0.339955,0.252595,0.005525
9,shared_mlp,balanced,0.145056,0.013238,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,1.000000,0.000000,0.258570,0.057676


,comparison_group,reference,method,delta_ap_mean,delta_ap_ci_low,delta_ap_bootstrap_median,delta_ap_ci_high,bootstrap_repeats,bootstrap_kind
0,lsr_latent_mlp,lsr_nn_none,lsr_nn_balanced,0.109347,-0.045587,0.102917,0.259209,30,paired IID rows; same resample across seeds; c...
1,lsr_latent_mlp,lsr_nn_none,lsr_nn_weighted,0.030137,0.010464,0.037288,0.077611,30,paired IID rows; same resample across seeds; c...
2,lsr_raw_lightgbm,lsr_lgbm_none,lsr_lgbm_balanced,-0.003844,-0.071401,0.005898,0.124697,30,paired IID rows; same resample across seeds; c...
3,lsr_raw_lightgbm,lsr_lgbm_none,lsr_lgbm_weighted,0.000000,0.000000,0.000000,0.000000,30,paired IID rows; same resample across seeds; c...
4,normalized_head_mlp,norm_ce,ldam,-0.078437,-0.260027,-0.082041,0.072014,30,paired IID rows; same resample across seeds; c...
5,normalized_head_mlp,norm_ce,ldam_drw,-0.095986,-0.250884,-0.090791,0.022167,30,paired IID rows; same resample across seeds; c...
6,shared_mlp,ce,balanced,0.001236,-0.017490,-0.001047,0.036683,30,paired IID rows; same resample across seeds; c...
7,shared_mlp,ce,cb_ce,0.002466,-0.018774,-0.002392,0.046283,30,paired IID rows; same resample across seeds; c...
8,shared_mlp,ce,cb_focal,0.003974,-0.021918,0.006188,0.045797,30,paired IID rows; same resample across seeds; c...
9,shared_mlp,ce,drw,0.002885,-0.017346,0.001535,0.039750,30,paired IID rows; same resample across seeds; c...


,comparison_group,method,ap_mean,ap_std,recall_frozen_threshold_mean,recall_frozen_threshold_std,fpr_frozen_threshold_mean,fpr_frozen_threshold_std,precision_frozen_threshold_mean,precision_frozen_threshold_std,older_fpr_mean,older_fpr_std,younger_fpr_mean,younger_fpr_std,fpr_gap_older_minus_younger_mean,fpr_gap_older_minus_younger_std,fpr_ratio_min_max_mean,fpr_ratio_min_max_std,fit_seconds_mean,fit_seconds_std,paper,paper_year,author_repo,author_commit,mode,implementation_scope,original_paper_score,split,seeds,evidence_path,interpretation,reference,delta_ap_mean,delta_ap_ci_low,delta_ap_bootstrap_median,delta_ap_ci_high,bootstrap_repeats,bootstrap_kind
0,lsr_latent_mlp,lsr_nn_balanced,0.302812,0.047124,0.138889,0.117851,0.083333,0.103986,0.350000,0.212132,0.067308,0.095187,0.10,0.113137,-0.032692,0.017950,0.373932,0.528819,0.792004,0.020533,Within-backbone control; not an independent paper,NaN,,,synthetic,Control,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42, 317]",/workspace/scratch/a3538503433e/work/research1...,Observed BAF result; no claim of reproducing o...,lsr_nn_none,0.109347,-0.045587,0.102917,0.259209,30.0,paired IID rows; same resample across seeds; c...
1,lsr_latent_mlp,lsr_nn_none,0.193465,0.079187,0.000000,0.000000,0.004902,0.006932,0.000000,0.000000,0.009615,0.013598,0.00,0.000000,0.009615,0.013598,0.500000,0.707107,0.734251,0.007685,Within-backbone control; not an independent paper,NaN,,,synthetic,Control,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42, 317]",/workspace/scratch/a3538503433e/work/research1...,Observed BAF result; no claim of reproducing o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lsr_latent_mlp,lsr_nn_weighted,0.223602,0.039173,0.000000,0.000000,0.009804,0.013865,0.000000,0.000000,0.019231,0.027196,0.00,0.000000,0.019231,0.027196,0.500000,0.707107,0.792903,0.131122,Latent Score-Based Reweighting for Robust Clas...,2025.0,https://github.com/YunzeTong/latent-score-base...,f8561071b72942e0001612bfea505aaeadbb4e73,synthetic,Budgeted BAF transfer,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42, 317]",/workspace/scratch/a3538503433e/work/research1...,Observed BAF result; no claim of reproducing o...,lsr_nn_none,0.030137,0.010464,0.037288,0.077611,30.0,paired IID rows; same resample across seeds; c...
3,lsr_raw_lightgbm,lsr_lgbm_balanced,0.275245,0.000872,0.138889,0.196419,0.049020,0.027730,0.208333,0.294628,0.057692,0.027196,0.04,0.028284,0.017692,0.001088,0.650000,0.183848,0.043765,0.004256,Within-backbone control; not an independent paper,NaN,,,synthetic,Control,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42, 317]",/workspace/scratch/a3538503433e/work/research1...,Observed BAF result; no claim of reproducing o...,lsr_lgbm_none,-0.003844,-0.071401,0.005898,0.124697,30.0,paired IID rows; same resample across seeds; c...
4,lsr_raw_lightgbm,lsr_lgbm_none,0.279089,0.012928,0.083333,0.117851,0.029412,0.041595,0.166667,0.235702,0.019231,0.027196,0.04,0.056569,-0.020769,0.029372,0.740385,0.367152,0.076057,0.065393,Within-backbone control; not an independent paper,NaN,,,synthetic,Control,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42, 317]",/workspace/scratch/a3538503433e/work/research1...,Observed BAF result; no claim of reproducing o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,lsr_raw_lightgbm,lsr_lgbm_weighted,0.279089,0.012928,0.083333,0.117851,0.029412,0.041595,0.166667,0.235702,0.019231,0.027196,0.04,0.056569,-0.020769,0.029372,0.740385,0.367152,0.035934,0.004332,Latent Score-Based Reweighting for Robust Clas...,2025.0,https://github.com/YunzeTong/latent-score-base...,f8561071b72942e0001612bfea505aaeadbb4e73,synthetic,Budgeted BAF transfer,Not copied; read the paper and report separately,train0-5 minus balanced meta; validation6; dev...,"[42

## 3. Xuất proof đánh giá

    Đây là proof của giai đoạn đánh giá. Giữ thêm full proof của12–15 để có toàn bộ checkpoint/training. Review ZIP có score/nhãn/code/output và chỉ xác minh các file được giữ. Chưa chạy CCRW-E hay cải tiến đề xuất trong gói khảo sát này.

In [6]:
PROOF = gru.research_export_proof(EVALUATION, ROOT)
for kind, path in PROOF.items():
    print(kind, "→", path)


RESEARCH_TEST_TABLE_VERIFIED 40
FULL_PROOF /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/evaluation_ccaacbcb_full_proof.zip
PARTIAL_REVIEW /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/evaluation_ccaacbcb_review.zip
full_proof → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/evaluation_ccaacbcb_full_proof.zip
review → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/evaluation_ccaacbcb_review.zip
